In [5]:
import digitalhub as dh

dh.get_current_profile()

'__default'

In [9]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # Carica le variabili d'ambiente dal file .env


# Inizializza il client (assicurati di aver impostato la variabile d'ambiente OPENAI_API_KEY)
client = OpenAI()


In [10]:
from pathlib import Path
from contextlib import ExitStack

folder = Path("./file_to_load")
all_files = sorted(p for p in folder.iterdir() if p.is_file())
file_names = [p.name for p in all_files]
print("File trovati:", file_names)

# Only for test: first  5 PDF 
pdf_paths = [p for p in all_files if p.suffix.lower() == ".pdf"][:5]
if not pdf_paths:
    raise FileNotFoundError(f"Nessun PDF trovato in {folder.resolve()}")

print("PDF selezionati:", [p.name for p in pdf_paths])



File trovati: ['Modulo1_Lezione1-1_Bibliografia.pdf', 'Modulo1_Lezione1-1_Caso1.pdf', 'Modulo1_Lezione1-1_Caso2.pdf', 'Modulo1_Lezione1-1_Guida.pdf', 'Modulo1_Lezione1-1_Materiale.pdf', 'Modulo1_Lezione1-1_Slide1.pdf', 'Modulo1_Lezione1-1_Slide2.pdf', 'Modulo1_Lezione1-2_Bibliografia.pdf', 'Modulo1_Lezione1-2_Caso1.pdf', 'Modulo1_Lezione1-2_Caso2.pdf', 'Modulo1_Lezione1-2_Guida.pdf', 'Modulo1_Lezione1-2_Materiale.pdf', 'Modulo1_Lezione1-3_Bibliografia.pdf', 'Modulo1_Lezione1-3_Caso1.pdf', 'Modulo1_Lezione1-3_Caso2.pdf', 'Modulo1_Lezione1-3_Guida.pdf', 'Modulo1_Lezione1-3_Materiale.pdf', 'Modulo2_Lezione2-1_Caso.pdf', 'Modulo2_Lezione2-1_Materiale.pdf', 'Modulo2_Lezione2-1_Slide1.pdf', 'Modulo2_Lezione2-1_Slide2.pdf', 'Modulo2_Lezione2-2_Bibliografia.pdf', 'Modulo2_Lezione2-2_Caso1.pdf', 'Modulo2_Lezione2-2_Caso2.pdf', 'Modulo2_Lezione2-2_Materiale.pdf', 'Modulo2_Lezione2-2_Slide1.pdf', 'Modulo2_Lezione2-2_Slide2.pdf', 'Modulo2_Lezione2-2_Slide3.pdf', 'Modulo2_Lezione2-3_Bibliografia.pd

In [17]:
def get_or_create_vector_store(client, name="spg-Core"):
    for store in client.vector_stores.list():
        if store.name == name:
            print(f"Vector store esistente: {store.id}")
            return store

    store = client.vector_stores.create(name=name)
    print(f"Vector store creato: {store.id}")
    return store


vector_store = get_or_create_vector_store(client)
vector_store_id = vector_store.id

Vector store creato: vs_6aedaa94d6008191b47c3d5592fe3bfc


In [19]:

with ExitStack() as stack:
    file_streams = [stack.enter_context(path.open("rb")) for path in pdf_paths]
    batch = client.vector_stores.file_batches.upload_and_poll(
        vector_store_id=vector_store_id,
        files=file_streams,
    )

print("Stato batch:", batch.status)
print("File:", batch.file_counts)

if batch.status != "completed":
    raise RuntimeError(f"Caricamento non completato: {batch.status}")


Stato batch: completed
File: FileCounts(cancelled=0, completed=5, failed=0, in_progress=0, total=5)


In [20]:
prompt = (
    "Sei un assistente di ricerca. Rispondi alle domande basandoti esclusivamente sui file caricati. "
    "Se non trovi la risposta nei file, dichiaralo chiaramente."
)
question = "descrivi per ciascun file il contenuto e le informazioni principali, indicandone le pagine di riferimento"
response = client.responses.create(
        model="gpt-4o",
        instructions=prompt,
        input=question,
        tools=[
            {
                "type": "file_search",
                "vector_store_ids": [vector_store.id],
            }
        ],

    )
print("ID della risposta:", response.id)
print("Stato della risposta:", response.status)
print("risposta:", response.output_text)


ID della risposta: resp_05183bb1b99e6fc3006aedae7e936c87d2bf2ba8f151e59369
Stato della risposta: completed
risposta: Ecco una descrizione del contenuto e delle informazioni principali per ciascun file:

1. **Modulo1_Lezione1-1_Bibliografia.pdf**
   - **Contenuto Principale**: Il file contiene una bibliografia che include riferimenti a studi e manuali diagnostici riguardanti le dipendenze comportamentali, in particolare l'uso problematico di internet e smartphone. Sono presenti strategie per la documentazione sistematica delle osservazioni scolastiche e protocolli per il dialogo con le famiglie.
   - **Pagine di Riferimento**: La bibliografia è delineata chiaramente, con un focus su risorse utili per approfondire queste tematiche.

2. **Modulo1_Lezione1-1_Caso2.pdf**
   - **Contenuto Principale**: Il documento esplora il caso di Marco e come la scuola e la famiglia possono intervenire in situazioni di uso problematico delle tecnologie digitali. Include dati e report da studi come ESPAD,